In [ ]:
!pip install datasets transformers evaluate jiwer torch gdown -q

In [ ]:
import os
import tarfile
import torch
import numpy as np
import datasets
import evaluate

from dataclasses import dataclass
from typing import Any, Dict, List, Union

from datasets import load_dataset, Audio
from transformers import (
    WhisperFeatureExtractor,
    WhisperTokenizer,
    WhisperProcessor,
    WhisperForConditionalGeneration,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments
)

In [ ]:
import gdown

url = "https://drive.google.com/uc?id=17I3oHLlGDwqWCfB3U0OBpxsh0xYN1VeX"
output = "dataset.tar.gz"

gdown.download(url, output, quiet=False)

with tarfile.open(output) as tar:
    tar.extractall("data")

Downloading...
From (original): https://drive.google.com/uc?id=17I3oHLlGDwqWCfB3U0OBpxsh0xYN1VeX
From (redirected): https://drive.google.com/uc?id=17I3oHLlGDwqWCfB3U0OBpxsh0xYN1VeX&confirm=t&uuid=3cea7f8e-1dd0-4952-a11b-f1208a2589be
To: /content/dataset.tar.gz
100%|██████████| 167M/167M [00:01<00:00, 102MB/s]
/tmp/ipykernel_15899/1435071630.py:9: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall("data")


In [ ]:
base_path = "data/cv-corpus-25.0-2026-03-09/as"

dataset = load_dataset(
    "csv",
    data_files={
        "train": f"{base_path}/train.tsv",
        "test": f"{base_path}/test.tsv"
    },
    delimiter="\t"
)

In [ ]:
def add_audio_path(batch):
    batch["audio"] = f"{base_path}/clips/{batch['path']}"
    return batch

dataset = dataset.map(add_audio_path)

dataset = dataset.cast_column("audio", Audio(sampling_rate=16000))

train_ds = dataset["train"]
test_ds = dataset["test"].select(range(10))  # required

In [ ]:
language = "as"

feature_extractor = WhisperFeatureExtractor.from_pretrained("openai/whisper-tiny")

tokenizer = WhisperTokenizer.from_pretrained(
    "openai/whisper-tiny",
    language=language,
    task="transcribe"
)

processor = WhisperProcessor.from_pretrained(
    "openai/whisper-tiny",
    language=language,
    task="transcribe"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
def prepare_dataset(batch):
    audio = batch["audio"]

    # safer check
    if audio is None:
        return {"input_features": None, "labels": None}

    try:
        input_features = feature_extractor(
            audio["array"],
            sampling_rate=audio["sampling_rate"]
        ).input_features[0]
    except Exception:
        return {"input_features": None, "labels": None}

    labels = tokenizer(batch["sentence"]).input_ids

    return {
        "input_features": input_features,
        "labels": labels
    }

train_ds = train_ds.map(prepare_dataset)
test_ds = test_ds.map(prepare_dataset)

train_ds = train_ds.filter(lambda x: x["input_features"] is not None)
test_ds = test_ds.filter(lambda x: x["input_features"] is not None)

train_ds = train_ds.remove_columns(
    [col for col in train_ds.column_names if col not in ["input_features", "labels"]]
)

test_ds = test_ds.remove_columns(
    [col for col in test_ds.column_names if col not in ["input_features", "labels"]]
)

print("Sample:", train_ds[0])
print("Columns:", train_ds.column_names)

Sample: {'input_features': [[-0.5712486505508423, -0.5712486505508423, -0.5712486505508423, -0.5712486505508423, -0.5712486505508423, -0.5712486505508423, -0.5712486505508423, -0.5712486505508423, -0.5712486505508423, -0.5712486505508423, -0.5712486505508423, -0.5712486505508423, -0.5712486505508423, -0.5712486505508423, -0.5712486505508423, -0.5712486505508423, -0.5712486505508423, -0.5712486505508423, -0.5712486505508423, -0.5712486505508423, -0.5712486505508423, -0.5712486505508423, -0.5712486505508423, -0.5712486505508423, -0.5712486505508423, -0.5712486505508423, -0.5712486505508423, -0.5712486505508423, -0.4887353181838989, -0.5600197315216064, -0.5712486505508423, -0.5712486505508423, -0.5712486505508423, -0.5712486505508423, -0.5712486505508423, -0.5712486505508423, -0.5712486505508423, -0.5712486505508423, -0.5712486505508423, -0.5712486505508423, -0.5712486505508423, -0.5712486505508423, -0.5712486505508423, -0.5712486505508423, -0.5712486505508423, -0.5712486505508423, -0.57

In [ ]:
model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-tiny")

model.config.language = "as"
model.config.task = "transcribe"
model.config.forced_decoder_ids = None

model.config.use_cache = False


Loading weights:   0%|          | 0/167 [00:00<?, ?it/s]

In [ ]:
@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any

    def __call__(self, features: List[Dict[str, Union[List[int], np.ndarray]]]):

        input_features = [{"input_features": f["input_features"]} for f in features]
        label_features = [{"input_ids": f["labels"]} for f in features]

        batch = self.processor.feature_extractor.pad(
            input_features, return_tensors="pt"
        )

        labels_batch = self.processor.tokenizer.pad(
            label_features, return_tensors="pt"
        )

        labels = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1), -100
        )

        batch["labels"] = labels
        return batch

data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)


In [ ]:
import evaluate

metric = evaluate.load("wer")

def compute_metrics(pred):
    pred_ids = pred.predictions

    # handle tuple outputs
    if isinstance(pred_ids, tuple):
        pred_ids = pred_ids[0]

    # handle logits → token IDs (safety)
    if isinstance(pred_ids, np.ndarray) and pred_ids.ndim == 3:
        pred_ids = np.argmax(pred_ids, axis=-1)

    label_ids = pred.label_ids
    label_ids[label_ids == -100] = tokenizer.pad_token_id

    # decode
    pred_str = tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = tokenizer.batch_decode(label_ids, skip_special_tokens=True)

    # compute WER
    wer = metric.compute(predictions=pred_str, references=label_str)

    return {"wer": wer}

In [ ]:
training_args = Seq2SeqTrainingArguments(
    output_dir="./whisper-assamese",
    per_device_train_batch_size=8,   # ✅ required
    per_device_eval_batch_size=1,    # ✅ required
    learning_rate=1e-5,
    warmup_steps=50,                # ✅ required
    max_steps=100,                  # ✅ required
    eval_strategy="steps",
    save_steps=100,                 # ✅ required
    eval_steps=100,                 # ✅ required
    logging_steps=25,
    predict_with_generate=True,
    generation_max_length=225,
    fp16=torch.cuda.is_available(),
    save_total_limit=2,
    report_to="none",
    remove_unused_columns=False     # critical fix
)

In [ ]:
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

In [ ]:
trainer.train()

Step,Training Loss,Validation Loss,Wer
100,1.415071,1.427971,3.432099


Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
Transcription using a multilingual Whisper will default to language detection followed by transcription instead of translation to English. This might be a breaking change for your use case. If you want to instead always translate your audio to English, make sure to pass `language='en'`. See https://github.com/huggingface/transformers/pull/28687 for more details.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.log

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=100, training_loss=1.9524123001098632, metrics={'train_runtime': 151.4537, 'train_samples_per_second': 5.282, 'train_steps_per_second': 0.66, 'total_flos': 1.9695108096e+16, 'train_loss': 1.9524123001098632, 'epoch': 0.8333333333333334})

In [ ]:
example = train_ds[55]  # 56th example (0-indexed)
print(example["labels"][0])

50258


In [ ]:
token_id = 51833

# Convert ID → token
token = tokenizer.convert_ids_to_tokens(token_id)

print("Token ID:", token_id)
print("Corresponding token:", token)

Token ID: 51833
Corresponding token: <|29.38|>


In [ ]:
is_special = token in tokenizer.all_special_tokens

print("Token ID:", token_id)
print("Token:", token)
print("Is special token?:", is_special)

Token ID: 51833
Token: <|29.38|>
Is special token?: False


In [ ]:
token_id = 50350

token = tokenizer.convert_ids_to_tokens(token_id)

print("Token ID:", token_id)
print("Corresponding token:", token)

Token ID: 50350
Corresponding token: <|as|>
